# Toy Topic Classification with compression_knn

This notebook exercises the repository's vectorized implementation of the paper's compression-based kNN classifier on a small topic-classification dataset.

It focuses on three paper-aligned checks:
- basic topic classification
- few-shot behavior
- compressor choice

In [20]:
import importlib
import sys
from pathlib import Path

import numpy as np

workspace_root = Path.cwd()
if not (workspace_root / "compression_knn").exists():
    workspace_root = workspace_root.parent
sys.path.insert(0, str(workspace_root))

import compression_knn
import compression_knn.knn as knn_module
import compression_knn.utils as utils_module

importlib.reload(knn_module)
importlib.reload(utils_module)
importlib.reload(compression_knn)

from compression_knn import CompressionKNNClassifier
from compression_knn.utils import compression_length

np.set_printoptions(precision=3, suppress=True)
print(f"workspace_root={workspace_root}")

workspace_root=d:\projects\random_projects\compression-knn


In [21]:
# 1. Build a Toy Topic Dataset
# The paper studies topic classification, so this notebook uses
# short texts from four topics with clear lexical overlap.

In [22]:
toy_train = {
    "sports": [
        "sports team wins the football match",
        "sports coach reviews the soccer match",
        "sports player scores a goal in the match",
        "sports league opens with a football team victory",
        "sports fans celebrate the goal and the team win",
    ],
    "technology": [
        "technology team ships python software code",
        "technology engineers deploy database software",
        "technology code updates the cloud server",
        "technology software team fixes the python bug",
        "technology platform stores data in the database",
    ],
    "cooking": [
        "cooking recipe says bake the cake slowly",
        "cooking chef stirs the sauce in the kitchen",
        "cooking recipe adds herbs to the soup",
        "cooking kitchen heat helps the sauce simmer",
        "cooking baker mixes flour before the bake",
    ],
    "business": [
        "business revenue growth helps the market outlook",
        "business investors discuss profit and sales",
        "business company reports strong quarterly revenue",
        "business market strategy improves profit guidance",
        "business sales team closes the investor meeting",
    ],
}

toy_test = {
    "sports": [
        "sports team scores the winning goal",
        "sports coach studies the football match",
    ],
    "technology": [
        "technology software code fixes the server bug",
        "technology team moves data into the database",
    ],
    "cooking": [
        "cooking recipe tells the chef to bake slowly",
        "cooking kitchen sauce needs more herbs",
    ],
    "business": [
        "business revenue and profit lift the market",
        "business investors praise the sales strategy",
    ],
}


def flatten_dataset(dataset):
    samples = []
    labels = []
    for label, texts in dataset.items():
        samples.extend(texts)
        labels.extend([label] * len(texts))
    return np.array(samples, dtype=str), np.array(labels, dtype=str)


X_train, y_train = flatten_dataset(toy_train)
X_test, y_test = flatten_dataset(toy_test)

print(f"train shape={X_train.shape}, test shape={X_test.shape}")
print(f"classes={sorted(set(y_train))}")

train shape=(20,), test shape=(8,)
classes=[np.str_('business'), np.str_('cooking'), np.str_('sports'), np.str_('technology')]


In [23]:
# 2. Fit the Vectorized Classifier and Inspect Distances
# The estimator already implements the paper's method, so this
# cell group uses the repo implementation directly.

In [24]:
classifier = CompressionKNNClassifier(
    n_neighbors=1,
    compressor="gzip",
    random_state=0,
)
classifier.fit(X_train, y_train)

predictions = classifier.predict(X_test)
accuracy = np.mean(predictions == y_test)
distance_matrix = classifier._distance_matrix(X_test)

print(f"accuracy={accuracy:.3f}")
print(f"distance_matrix shape={distance_matrix.shape}")

for class_name in np.unique(y_test):
    class_mask = y_test == class_name
    class_accuracy = np.mean(predictions[class_mask] == y_test[class_mask])
    print(f"{class_name:10s} class_accuracy={class_accuracy:.3f}")

print("\nNearest neighbors for the first three test samples:")
for test_index, test_text in enumerate(X_test[:3]):
    nearest = np.argsort(distance_matrix[:, test_index])[:3]
    neighbor_labels = y_train[nearest]
    neighbor_distances = distance_matrix[nearest, test_index]
    print(f"test[{test_index}]={test_text}")
    print(f"  predicted={predictions[test_index]} true={y_test[test_index]}")
    print(f"  neighbor_labels={neighbor_labels.tolist()}")
    print(f"  neighbor_distances={neighbor_distances.tolist()}")

accuracy=1.000
distance_matrix shape=(20, 8)
business   class_accuracy=1.000
cooking    class_accuracy=1.000
sports     class_accuracy=1.000
technology class_accuracy=1.000

Nearest neighbors for the first three test samples:
test[0]=sports team scores the winning goal
  predicted=sports true=sports
  neighbor_labels=['sports', 'sports', 'sports']
  neighbor_distances=[0.42, 0.42592592592592593, 0.42857142857142855]
test[1]=sports coach studies the football match
  predicted=sports true=sports
  neighbor_labels=['sports', 'sports', 'sports']
  neighbor_distances=[0.32727272727272727, 0.34545454545454546, 0.39285714285714285]
test[2]=technology software code fixes the server bug
  predicted=technology true=technology
  neighbor_labels=['technology', 'technology', 'technology']
  neighbor_distances=[0.2833333333333333, 0.31666666666666665, 0.3333333333333333]


In [25]:
# 3. Few-Shot Sweep
# Keep the test set fixed and vary the number of labeled examples
# per class to mimic the paper's low-resource setting.

In [26]:
few_shot_results = []

for shots_per_class in [1, 2, 3, 5]:
    few_shot_train = {
        label: texts[:shots_per_class]
        for label, texts in toy_train.items()
    }
    X_few, y_few = flatten_dataset(few_shot_train)
    few_shot_classifier = CompressionKNNClassifier(
        n_neighbors=1,
        compressor="gzip",
        random_state=0,
    )
    few_shot_classifier.fit(X_few, y_few)
    few_shot_accuracy = np.mean(few_shot_classifier.predict(X_test) == y_test)
    few_shot_results.append((shots_per_class, len(X_few), few_shot_accuracy))

print("shots_per_class total_train accuracy")
for shots_per_class, total_train, few_shot_accuracy in few_shot_results:
    print(f"{shots_per_class:15d} {total_train:11d} {few_shot_accuracy:.3f}")

shots_per_class total_train accuracy
              1           4 1.000
              2           8 1.000
              3          12 1.000
              5          20 1.000


In [27]:
joined_train = np.array([" ".join(X_train.tolist())], dtype=str)
original_bytes = len(joined_train[0].encode("utf-8"))

print("compressor accuracy compression_ratio")
for compressor_name in ["gzip", "bzip2", "lzma"]:
    compressor_classifier = CompressionKNNClassifier(
        n_neighbors=1,
        compressor=compressor_name,
        random_state=0,
    )
    compressor_classifier.fit(X_train, y_train)
    compressor_accuracy = np.mean(
        compressor_classifier.predict(X_test) == y_test
    )
    compressed_bytes = int(compression_length(joined_train, compressor_name)[0])
    compression_ratio = original_bytes / compressed_bytes
    print(
        f"{compressor_name:10s} {compressor_accuracy:.3f} {compression_ratio:.3f}"
    )

compressor accuracy compression_ratio
gzip       1.000 2.164
bzip2      1.000 2.007
lzma       1.000 1.742


## 4. Compare Compressors

The paper compares compressors. The repo exposes the same choices through the estimator's `compressor` parameter.